# 1 — Audit an annotated atlas

`celltype-audit` treats a cell type's **label** and its **expression** as two independent
claims about the same cells, and reports where they disagree. It does not re-annotate
an atlas; it tells you where the atlas disagrees with itself.

You need an `.h5ad` with cell-type labels and a tissue. Anything CELLxGENE-standardised
already has both.

In [ ]:
!pip install -q celltype-audit

In [ ]:
from celltype_audit import audit_h5ad

report = audit_h5ad(
    "TS-Lung.h5ad",
    organ="Lung",              # helps resolve labels that drop the organ CL keeps
    # tissue='UBERON:0002048'  # only needed if the file carries no tissue id
)
print(report.summary())

## The two outputs mean different things

**`report.flagged`** — the lineage sweep. High precision (~83%), few results.
Each is a claim worth checking: the label's lineage is disjoint from every candidate
the markers support.

**`report.queue`** — ranked triage, *not* assertions. Work down it until the hit rate
stops justifying the time. In the reference study the top 5 were 60% real against a
7.9% base rate.

In [ ]:
for r in report.flagged:
    print("%-32s %7d  %-26s -> %s" % (
        r["atlas_label"], r["n_cells"],
        r["assignment"]["label"], r["audit"]["best_term_label"]))

In [ ]:
for r in report.queue[:10]:
    a = r['audit']
    print("%-32s ratio %.2f  %-28s  %s" % (
        r['atlas_label'], a['ratio'] or 0, a['best_term_label'],
        ', '.join(r['evidence']['markers'][:4])))

## Reading one record

`related_to_best` false means the winner is not merely a coarser or finer name.
`thin_support` false means it is not resting on a poorly-estimated profile.

`ratio` near 1 means the asserted term *also* fits — which happens between genuinely
similar cell types — so **the identity of the winner is more trustworthy than the
margin**.

In [ ]:
import json
print(json.dumps(report.queue[0], indent=1)[:900])

In [ ]:
report.to_json('annotations.json')
report.to_tsv('annotations.tsv')